# Token Usage & Cost Analysis with DistLLM

This notebook tracks token usage and estimated cost across several requests using:

- `response.usage` from each chat completion (prompt/completion/total tokens, and   an optional `cost_usd`), and
- the DistLLM SDK client **stats** aggregator (`estimate_cost`), which accumulates   totals across all calls.

Finally it plots a simple bar chart with `matplotlib`.

## Requirements

```bash
pip install openai distllm_sdk matplotlib
```

Start the DistLLM API server before running the code cells (see the README).

In [ ]:
# ---------------------------------------------------------------------------
# Connection configuration
# ---------------------------------------------------------------------------
# Point the OpenAI-compatible client at a running DistLLM API.
# Start the server first, e.g.:
#     distllm-api --model distributed-llm --host 0.0.0.0 --port 8000
#
# The API key is not required for a local deployment; set DISTLLM_API_KEY if
# your deployment enforces authentication.
import os

BASE_URL = os.environ.get("DISTLLM_BASE_URL", "http://localhost:8000/v1")
API_KEY = os.environ.get("DISTLLM_API_KEY", "not-needed")
MODEL = os.environ.get("DISTLLM_MODEL", "distributed-llm")

print(f"Using endpoint: {BASE_URL}")
print(f"Model: {MODEL}")


In [ ]:
# ---------------------------------------------------------------------------
# Run a set of representative requests and collect usage
# ---------------------------------------------------------------------------
try:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
except ImportError:
    from distllm_sdk.compat import openai_compat as oc
    client = oc.OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Each entry is a short label + a prompt of differing length/cost profile.
scenarios = [
    ('short-q',  'What is DistLLM?'),
    ('medium-q', 'Explain how DistLLM shards a model across consumer GPUs, in 3 sentences.'),
    ('long-q',   'Write a detailed paragraph comparing DistLLM to single-GPU inference, '
                 'covering latency, cost, and throughput.'),
    ('code-q',   'Show a 5-line Python snippet that calls the DistLLM chat API.'),
]

records = []  # (label, prompt_tokens, completion_tokens, cost_usd)
for label, prompt in scenarios:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=128, temperature=0.4)
        u = getattr(resp, 'usage', None)
        pt = u.prompt_tokens if u else 0
        ct = u.completion_tokens if u else 0
        cost = getattr(u, 'cost_usd', 0.0) or 0.0
        records.append((label, pt, ct, float(cost)))
        print(f'{label:8s} prompt={pt:4d} completion={ct:4d} cost=${cost:.6f}')
    except Exception as exc:  # pragma: no cover
        print(f'[!] request {label} failed: {exc}')
        records.append((label, 0, 0, 0.0))


In [ ]:
# ---------------------------------------------------------------------------
# Aggregate with the DistLLM SDK stats (optional but handy)
# ---------------------------------------------------------------------------
# distllm_sdk's DistLLMClientSync accumulates per-call stats you can turn into a
# cost estimate. We guard it so the notebook still runs if the SDK is absent.
try:
    from distllm_sdk import DistLLMClientSync
    sdk = DistLLMClientSync(base_url='http://localhost:8000')
    for label, prompt in scenarios:
        try:
            sdk.chat_completions(messages=[{'role': 'user', 'content': prompt}], max_tokens=128)
        except Exception:
            pass
    stats = sdk.stats()
    # Default pricing: $0.50 / 1M input, $1.50 / 1M output tokens.
    est = stats.estimate_cost()
    print(f'SDK aggregate -> prompt={stats.total_prompt_tokens} '
          f'completion={stats.total_completion_tokens} est_cost=${est:.6f}')
except Exception as exc:  # pragma: no cover
    print(f'SDK stats unavailable ({exc}); using per-response usage only.')
    stats = None


In [ ]:
# ---------------------------------------------------------------------------
# Plot a bar chart of token usage and cost
# ---------------------------------------------------------------------------
try:
    import matplotlib.pyplot as plt
    import numpy as np

    if not records:
        raise RuntimeError('No usage records collected.')

    labels = [r[0] for r in records]
    prompt_tok = np.array([r[1] for r in records], dtype=float)
    compl_tok = np.array([r[2] for r in records], dtype=float)
    costs = np.array([r[3] for r in records], dtype=float)

    x = np.arange(len(labels))
    width = 0.38

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    # Left: stacked token counts
    ax1.bar(x - width/2, prompt_tok, width, label='prompt')
    ax1.bar(x + width/2, compl_tok, width, label='completion')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.set_ylabel('tokens')
    ax1.set_title('Token usage per scenario')
    ax1.legend()

    # Right: cost per scenario
    ax2.bar(x, costs, color='#2a9d8f')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.set_ylabel('estimated cost (USD)')
    ax2.set_title('Estimated cost per scenario')
    for i, c in enumerate(costs):
        ax2.text(i, c, f'${c:.5f}', ha='center', va='bottom', fontsize=8)

    fig.tight_layout()
    plt.show()
except Exception as exc:  # pragma: no cover
    print(f'\n[!] Plotting skipped: {exc}')
    print('    Install matplotlib to render the charts: pip install matplotlib')
